In [ ]:
# from dotenv import load_dotenv
from google.colab import userdata

from datasets import load_dataset

# HF_TOKEN = os.getenv("HF_TOKEN")
# WANDB_API_KEY = os.getenv("WANDB_API_KEY")

HF_TOKEN = userdata.get("HF_TOKEN")

In [ ]:
ds_id_001 = "rojagtap/bookcorpus"
ds_id_002 = "wikimedia/wikipedia"  # 20231101.en

In [ ]:
ds_001 = load_dataset(ds_id_001)
ds_001

In [ ]:
ds_id_002 = "wikimedia/wikipedia"
subset_id_002 = "20231101.en"

# Load the dataset in streaming mode
ds_002 = load_dataset(ds_id_002, subset_id_002)

In [ ]:
ds_002

In [ ]:
ds_id_003 = "ajaykarthick/imdb-movie-reviews"
ds_003 = load_dataset(ds_id_003)
ds_003

In [ ]:
ds_id_004 = "mteb/sts12-sts"
ds_id_005 = "mteb/sts13-sts"
ds_id_006 = "mteb/sts14-sts"
ds_id_007 = "mteb/sts15-sts"

ds_004 = load_dataset(ds_id_004)
ds_005 = load_dataset(ds_id_005)
ds_006 = load_dataset(ds_id_006)
ds_007 = load_dataset(ds_id_007)

In [ ]:
ds_007

In [ ]:
# ds_id_008 = "sentence-transformers/reddit"
# ds_008 = load_dataset(ds_id_008)
# ds_008

In [ ]:
from datasets import concatenate_datasets, DatasetDict

# ==========================================
# 1. STANDARDIZE DATASETS 1, 2, AND 3
# ==========================================

# ds_001: Just isolate the 'text' column
cleaned_001 = ds_001["train"].select_columns(["text"])

# ds_002: Keep only 'text'
cleaned_002 = ds_002["train"].select_columns(["text"])

# ds_003: Combine train/test, rename 'review' to 'text'
combined_003 = concatenate_datasets([ds_003["train"], ds_003["test"]])
cleaned_003 = combined_003.rename_column("review", "text").select_columns(["text"])


# ==========================================
# 2. LOOP THROUGH AND FLATTEN ds_004 TO ds_007
# ==========================================

sentence_datasets = [ds_004, ds_005, ds_006, ds_007]
cleaned_sentence_datasets = []

for idx, ds in enumerate(sentence_datasets, start=4):
    print(f"Processing ds_00{idx}...")

    # Combine the train and test splits for the current dataset
    if "train" in ds:
        combined = concatenate_datasets([ds["train"], ds["test"]])
    else:
        combined = ds["test"]
    # Flatten sentence1 and sentence2 into distinct rows under 'text'
    cleaned = combined.map(
        lambda batch: {"text": batch["sentence1"] + batch["sentence2"]},
        batched=True,
        remove_columns=combined.column_names,
    )
    cleaned_sentence_datasets.append(cleaned)

# select_008 = ds_008['train']
# cleaned_008 = select_008.map(
#     lambda batch: {
#         "text": [
#             f"{s1 or ''}\n{s2 or ''}"
#             for s1, s2 in zip(batch["title"], batch["body"])
#         ]
#     },
#     batched=True,
#     remove_columns=select_008.column_names
# )

# ==========================================
# 3. CONCATENATE EVERYTHING
# ==========================================

print("\nMerging all processed text streams...")
all_cleaned_parts = [cleaned_001, cleaned_002, cleaned_003] + cleaned_sentence_datasets
merged_dataset = concatenate_datasets(all_cleaned_parts)

print(f"Total rows compiled for BERT MLM: {len(merged_dataset):,}")


# ==========================================
# 4. THREE-WAY SPLIT (TRAIN / VAL / TEST)
# ==========================================

print("Creating train/val/test splits...")
seed = 17771
# Step 1: Split off the training data (90% train, 10% goes to a temporary holding split)
train_scratch, val_test_scratch = merged_dataset.train_test_split(
    test_size=0.10, seed=seed
).values()

# Step 2: Split the remaining 10% equally into validation and test (50/50 of the 10% = 5% each)
val_scratch, test_scratch = val_test_scratch.train_test_split(
    test_size=0.50, seed=seed
).values()

# Step 3: Bundle them up into a clean final DatasetDict
final_dataset_dict = DatasetDict(
    {"train": train_scratch, "validation": val_scratch, "test": test_scratch}
)

# Check the final structure
print("\nFinal Dataset Setup:")
print(final_dataset_dict)

In [ ]:
# Define your Hugging Face repository name
repo_id = "8Opt/bert-mlm-experiments-en"

print(f"Uploading dataset to Hugging Face Hub at: {repo_id}...")

# Push the dataset
final_dataset_dict.push_to_hub(
    repo_id=repo_id,
    # private=True,      # Set to False if you want it to be publicly visible
    max_shard_size="500MB",  # Breaks the huge 80M row dataset into stable 500MB files
)

print("Upload complete!")